# Decision Analysis 6 6


## Electre Tri-B


The objective of this laboratory session is to gain practical understanding of the **ELECTRE TRI-B** method – one of the most important _outranking-based sorting_ methods used in Multi-Criteria Decision Analysis (MCDA).

The ELECTRE TRI-B method is designed to assign a finite set of decision alternatives to predefined, ordered categories based on their performance with respect to multiple, often conflicting criteria. Instead of producing a complete ranking, the method focuses on classification by comparing alternatives with reference profiles that define the boundaries between categories.

Within the method, each alternative is compared to a set of boundary profiles that separate adjacent categories. For each criterion, the difference between the performance of the alternative and the profile is evaluated. This comparison is then used to construct a **concordance index**, which expresses the degree to which there is sufficient evidence to support the assertion that the alternative is at least as good as the profile.

To account for strong opposition on individual criteria, **discordance indices** are also computed. These indices capture situations where a significant disadvantage on a single criterion may weaken or even invalidate the overall outranking relation.

The concordance and discordance information are aggregated into a **credibility index**, which represents the strength of the outranking relation between the alternative and the profile. This index is then compared with a predefined cutting level ($\lambda$) to determine whether the outranking relation is validated.

The assignment procedure is performed using one of two possible rules:

- **pessimistic (conjunctive) assignment** – the alternative is assigned to the highest category for which it sufficiently outranks the lower boundary profile,
- **optimistic (disjunctive) assignment** – the alternative is assigned to the lowest category whose upper boundary profile does not sufficiently outrank the alternative.

In the ELECTRE TRI-B method, the decision maker provides preference information in the following form:

- **q** – indifference threshold,
- **p** – preference threshold,
- **v** – veto threshold,
- **w** – weight of criterion _k_,
- **$\lambda$** – cutting level for validating the outranking relation,
- direction of preference (cost or benefit criterion).

During the laboratory sessions, we will consider a simplified version of the ELECTRE method, in which a single set of preference information is defined instead of a separate set for each profile.

> **IMPORTANT**
>
> Code written in this notebook will be checked against automatic code checker and the points will be given based on its' results, please leave the function signatures unchanged.
> As a result there are no partial points for a programing tasks


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import utils

In [2]:
# Loading dataset and preference information
data_path = Path("")
dataset = utils.load_dataset(data_path)
boundary_profiles = utils.load_boundary_profiles(data_path)
preference_information = utils.load_preference_information(data_path)
credibility_threshold = 0.65

In [3]:
dataset

,g1,g2,g3,g4
Alternative,,,,
a1,90,86,46,30
a2,40,90,14,48
a3,94,100,40,36
a4,78,76,30,50
a5,60,60,30,30
a6,64,72,12,46
a7,62,88,22,48
a8,70,30,12,12


In [4]:
boundary_profiles

,g1,g2,g3,g4
Alternative,,,,
b1,64,61,32,32
b2,86,84,43,43


In [5]:
preference_information

,q,p,v,w,type
Criterion,,,,,
g1,2,6,20.0,40,gain
g2,2,5,24.0,30,gain
g3,0,2,NaN,25,gain
g4,0,2,NaN,5,gain


### Task 1 (maximum points: 1)

Implement the `difference_function` function.

$$
    d_j (a, b) = \left\{ \begin{array}{ll}
    g_j(a) - g_j(b), & \textrm{for \textit{gain} criterion},\\
    g_j(b) - g_j(a), & \textrm{for \textit{cost} criterion},\\
    \end{array} \right.
$$

This difference is used later to compute marginal concordance and discordance values.


In [ ]:
def difference_function(
    alternative_a: float, alternative_b: float, criterion_type: utils.CriterionType
) -> float:
    """
    Function that calculates the difference between given alternative pair on a single criterion.

    :param alternative_a: first alternative in a pair
    :param alternative_b: second alternative in a pair
    :param criterion_type: criterion type either gain or cost
    :return: difference between alternative pair calculated according to the criterion type
    """
    if criterion_type == utils.CriterionType.GAIN:
        return alternative_a - alternative_b
    elif criterion_type == utils.CriterionType.COST:
        return alternative_b - alternative_a
    else:
        raise ValueError("Invalid criterion type")

### Task 2 (maximum points: 1)

Implement the `calculate_marginal_concordance_index` function.

Requirements:

- Use inputs `diff`, `indifference_threshold` ($q$), and `preference_threshold` ($p$).
- Compute the marginal concordance value with the piecewise rule:

$$
c(a, b)=\begin{cases}
1, & d(a,b) \ge -q \\
0, & d(a,b) \le -p \\
\frac{p + d(a,b)}{p - q}, & -q > d(a,b) > -p
\end{cases}
$$


In [7]:
def calculate_marginal_concordance_index(
    diff: float, indifference_threshold: float, preference_threshold: float
) -> float:
    """
    Function that calculates the marginal concordance index for the given pair of alternatives, according to the formula presented during classes.

    :param diff: difference between compared alternatives either as a float for single criterion and alternative pairs, or as numpy array for multiple alternatives
    :param indifference_threshold: indifference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :param preference_threshold: preference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :return: marginal concordance index either as a float for single criterion and alternative pairs, or as numpy array for multiple criterion
    """
    if diff >= -indifference_threshold:
        return 1.0
    elif diff <= -preference_threshold:
        return 0.0
    else:
        return (preference_threshold + diff) / (
            preference_threshold - indifference_threshold
        )

### Task 3 (maximum points: 1)

Implement the `calculate_marginal_concordance_matrix` function.

Requirements:

- Use `dataset`, `boundary_profiles`, and `preference_information`.
- Build differences for both directions: alternative vs profile and profile vs alternative.
- Use `q` and `p` thresholds from `preference_information`.
- Return a 4D matrix with shape `[2, n_alternatives, n_profiles, n_criteria]`.


In [8]:
def calculate_marginal_concordance_matrix(
    dataset: pd.DataFrame,
    boundary_profiles: pd.DataFrame,
    preference_information: pd.DataFrame,
) -> np.ndarray:
    """
    Function that calculates the marginal concordance matrix for all alternatives pairs and criterion available in dataset

    :param dataset: pandas dataframe representing dataset with alternatives as rows and criterion as columns
    :param boundary_profiles: pandas dataframe with boundary profiles
    :param preference_information: pandas dataframe with preference information for all criterion
    :return: 4D numpy array with marginal concordance matrix with shape [2, number of alternatives, number of boundary profiles, number of criterion], where element with index [0, i, j, k] describe marginal concordance index between alternative i and boundary profile j on criterion k, while element with index [1, i, j, k] describe marginal concordance index between boundary profile j and  alternative i on criterion k
    """
    num_alternatives = dataset.shape[0]
    num_boundary_profiles = boundary_profiles.shape[0]
    num_criteria = dataset.shape[1]

    marginal_concordance_matrix = np.zeros(
        (2, num_alternatives, num_boundary_profiles, num_criteria)
    )
    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            for k in range(num_criteria):
                c_name = dataset.columns[k]
                c_type = preference_information.loc[
                    preference_information.index == c_name, "type"
                ].values[0]
                diff_a_b = difference_function(
                    dataset.iloc[i, k], boundary_profiles.iloc[j, k], c_type
                )
                diff_b_a = difference_function(
                    boundary_profiles.iloc[j, k], dataset.iloc[i, k], c_type
                )
                indifference_threshold = preference_information.loc[
                    preference_information.index == c_name,
                    "q",
                ].values[0]
                preference_threshold = preference_information.loc[
                    preference_information.index == c_name,
                    "p",
                ].values[0]
                marginal_concordance_matrix[0, i, j, k] = (
                    calculate_marginal_concordance_index(
                        diff_a_b, indifference_threshold, preference_threshold
                    )
                )
                marginal_concordance_matrix[1, i, j, k] = (
                    calculate_marginal_concordance_index(
                        diff_b_a, indifference_threshold, preference_threshold
                    )
                )

    return np.nan_to_num(marginal_concordance_matrix, nan=0.0)

In [9]:
marginal_concordance_matrix = calculate_marginal_concordance_matrix(
    dataset, boundary_profiles, preference_information
)

In [10]:
marginal_concordance_matrix[:, 4, 0]

array([[0.5, 1. , 0. , 0. ],
       [1. , 1. , 1. , 1. ]])

### Task 4 (maximum points: 1)

Implement the `calculate_comprehensive_concordance_matrix` function.

The function should aggregate marginal concordance values across all criteria using criterion weights.

For each pair of alternatives and boundary profile (a, b), compute the comprehensive concordance index:

$$C(a, b) = \frac{\sum w_k * c_k(a, b)}{\sum w_k}$$

where:

- $w_k$ is the weight of criterion k,
- $c_k(a, b)$ is the marginal concordance value.


In [11]:
def calculate_comprehensive_concordance_matrix(
    marginal_concordance_matrix: np.ndarray, preference_information: pd.DataFrame
) -> np.ndarray:
    num_alternatives = marginal_concordance_matrix.shape[1]
    num_boundary_profiles = marginal_concordance_matrix.shape[2]
    num_criteria = marginal_concordance_matrix.shape[3]

    comprehensive_concordance_matrix = np.zeros(
        (2, num_alternatives, num_boundary_profiles)
    )

    total_weight = sum(
        preference_information.loc[
            preference_information.index == dataset.columns[k], "w"
        ].values[0]
        for k in range(num_criteria)
    )

    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            concordance_a_b = 0.0
            concordance_b_a = 0.0
            for k in range(num_criteria):
                c_name = dataset.columns[k]
                weight = preference_information.loc[
                    preference_information.index == c_name, "w"
                ].values[0]
                concordance_a_b += weight * marginal_concordance_matrix[0, i, j, k]
                concordance_b_a += weight * marginal_concordance_matrix[1, i, j, k]

            # Divide by sum of weights
            comprehensive_concordance_matrix[0, i, j] = concordance_a_b / total_weight
            comprehensive_concordance_matrix[1, i, j] = concordance_b_a / total_weight

    return comprehensive_concordance_matrix

In [12]:
comprehensive_concordance_index = calculate_comprehensive_concordance_matrix(
    marginal_concordance_matrix, preference_information
)

In [13]:
print(comprehensive_concordance_index[0, 4, 1])  # C(a_i, b_i)
print(comprehensive_concordance_index[1, 4, 0])  # C(b_i, a_i)

0.0
1.0


### Task 5 (maximum points: 1)

Implement the `calculate_marginal_discordance_index` function.

Requirements:

- Use inputs `diff`, `ppreference_threshold` (p), and `veto threshold` (v).
- Compute the marginal discordance value with the piecewise rule:

$$
    D_j (a_i, b_{h}) = \left\{ \begin{array}{ll}
        1, & \textrm{if } d_k(a_{i}, b_{h}) \leq -v_j^h),\\
        0, & \textrm{if } d_k(a_{i}, b_{h}) \geq -p_j^h),\\
        \frac{-d_k(a_{i}, b_{h}) - p_j^h}{v_j^h - p_j^h}, & \textrm{if } -p_j^h > d_k(a_{i}, b_{h}) > -v_j^h).
    \end{array} \right.
$$


In [14]:
def calculate_marginal_discordance_index(
    diff: float, preference_threshold: float, veto_threshold: float
) -> float:
    """
    Function that calculates the marginal discordance index for the given pair of alternatives, according to the formula presented during classes.

    :param diff: difference between compared alternatives either as a float for single criterion and alternative pairs, or as numpy array for multiple alternatives
    :param preference_threshold: preference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :param veto_threshold: veto threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :return: marginal discordance index either as a float for single criterion and alternative pairs, or as numpy array for multiple criterion
    """
    if diff <= -veto_threshold:
        return 1.0
    elif diff >= -preference_threshold:
        return 0.0
    else:
        return (-diff - preference_threshold) / (veto_threshold - preference_threshold)

### Task 6 (maximum points: 1)

Implement the `calculate_marginal_discordance_matrix` function.

Requirements:

- Use `dataset`, `boundary_profiles`, `preference_thresholds`, `veto_thresholds`, and `criterion_types`.
- Build differences for both directions: alternative vs profile and profile vs alternative.
- Compute discordance values using `calculate_marginal_discordance_index`.
- Return a 4D matrix with shape `[2, n_alternatives, n_profiles, n_criteria]`.


In [15]:
def calculate_marginal_discordance_matrix(
    dataset: pd.DataFrame,
    boundary_profiles: pd.DataFrame,
    preference_thresholds,
    veto_thresholds,
    criterion_types,
) -> np.ndarray:
    """
    Function that calculates the marginal discordance matrix for all alternatives pairs and criterion available in dataset

    :param dataset: pandas dataframe representing dataset with alternatives as rows and criterion as columns
    :param boundary_profiles: pandas dataframe with boundary profiles
    :param preference_thresholds: pandas dataframe representing preference thresholds for all boundary profiles and criterion
    :param veto_thresholds: pandas dataframe representing veto thresholds for all boundary profiles and criterion
    :param criterion_types: pandas dataframe with a column 'type' representing the type of criterion (either gain or cost)
    :return: 4D numpy array with marginal discordance matrix with shape [2, number of alternatives, number of boundary profiles, number of criterion], where element with index [0, i, j, k] describe marginal discordance index between alternative i and boundary profile j on criterion k, while element with index [1, i, j, k] describe marginal discordance index between boundary profile j and  alternative i on criterion k
    """
    num_alternatives = dataset.shape[0]
    num_boundary_profiles = boundary_profiles.shape[0]
    num_criteria = dataset.shape[1]

    marginal_discordance_matrix = np.zeros(
        (2, num_alternatives, num_boundary_profiles, num_criteria)
    )

    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            for k in range(num_criteria):
                c_name = dataset.columns[k]
                c_type = criterion_types.loc[
                    criterion_types.index == c_name, "type"
                ].values[0]
                preference_threshold = preference_thresholds.loc[
                    preference_thresholds.index == c_name, "p"
                ].values[0]
                veto_threshold = veto_thresholds.loc[
                    veto_thresholds.index == c_name, "v"
                ].values[0]

                diff_a_b = difference_function(
                    dataset.iloc[i, k], boundary_profiles.iloc[j, k], c_type
                )
                diff_b_a = difference_function(
                    boundary_profiles.iloc[j, k], dataset.iloc[i, k], c_type
                )

                marginal_discordance_matrix[0, i, j, k] = (
                    calculate_marginal_discordance_index(
                        diff_a_b, preference_threshold, veto_threshold
                    )
                )
                marginal_discordance_matrix[1, i, j, k] = (
                    calculate_marginal_discordance_index(
                        diff_b_a, preference_threshold, veto_threshold
                    )
                )

    return np.nan_to_num(marginal_discordance_matrix, nan=0.0)

### Task 7 (maximum points: 1)

Implement the `calculate_credibility_index` function.

The function should compute the credibility index by combining the comprehensive concordance matrix with marginal discordance indices.

For each pair of alternatives and boundary profile (a, b), compute the credibility index:

$$\sigma(a, b) = C(a, b) \cdot \prod_{d_k(a, b) > C(a, b)} \frac{1 - d_k(a, b)}{1 - C(a, b)}$$

where:

- $C(a, b)$ is the comprehensive concordance index,
- $d_k(a, b)$ is the marginal discordance index on criterion k,
- the product is taken over criteria where discordance exceeds concordance.


In [16]:
def calculate_credibility_index(
    comprehensive_concordance_matrix: np.ndarray,
    marginal_discordance_matrix: np.ndarray,
) -> np.ndarray:
    """
    Function that calculates the credibility index for the given comprehensive concordance matrix and marginal discordance matrix

    :param comprehensive_concordance_matrix: 3D numpy array with comprehensive concordance matrix. Every entry in the matrix [i, j] represents comprehensive concordance index between alternative i and alternative j
    :param marginal_discordance_matrix: 3D numpy array with marginal discordance matrix, Consecutive indices [i, j, k] describe first alternative, second alternative, criterion
    :return: 3D numpy array with credibility matrix with shape [2, number of alternatives, number of boundary profiles], where element with index [0, i, j] describe credibility index between alternative i and boundary profile j, while element with index [1, i, j] describe credibility index between boundary profile j and  alternative i
    """
    num_alternatives = comprehensive_concordance_matrix.shape[1]
    num_boundary_profiles = comprehensive_concordance_matrix.shape[2]
    num_criteria = marginal_discordance_matrix.shape[3]

    credibility_matrix = np.zeros((2, num_alternatives, num_boundary_profiles))

    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            credibility_a_b = comprehensive_concordance_matrix[0, i, j]
            credibility_b_a = comprehensive_concordance_matrix[1, i, j]
            for k in range(num_criteria):
                if marginal_discordance_matrix[0, i, j, k] > credibility_a_b:
                    credibility_a_b *= (1 - marginal_discordance_matrix[0, i, j, k]) / (
                        1 - comprehensive_concordance_matrix[0, i, j]
                    )
                if marginal_discordance_matrix[1, i, j, k] > credibility_b_a:
                    credibility_b_a *= (1 - marginal_discordance_matrix[1, i, j, k]) / (
                        1 - comprehensive_concordance_matrix[1, i, j]
                    )
            credibility_matrix[0, i, j] = credibility_a_b
            credibility_matrix[1, i, j] = credibility_b_a

    return credibility_matrix

### Task 8 (maximum points: 1)

Implement the `calculate_outranking_relation_matrix` function.

The function should compare credibility values with the cutting level $\lambda$ (`credibility_threshold`) and return a boolean matrix indicating whether outranking holds for each pair.


In [17]:
def calculate_outranking_relation_matrix(
    credibility_index: np.ndarray, credibility_threshold: float
) -> np.ndarray:
    """
    Function that calculates boolean matrix with information if outranking holds for a given pair

    :param credibility_index: 3D numpy array with credibility matrix with shape [2, number of alternatives, number of boundary profiles], where element with index [0, i, j] describe credibility index between alternative i and boundary profile j, while element with index [1, i, j] describe credibility index between boundary profile j and  alternative i
    :param credibility_threshold: float number
    :return: 3D numpy boolean matrix with information if outranking holds for a given pair
    """
    num_alternatives = credibility_index.shape[1]
    num_boundary_profiles = credibility_index.shape[2]

    outranking_relation_matrix = np.zeros(
        (2, num_alternatives, num_boundary_profiles), dtype=bool
    )

    for i in range(num_alternatives):
        for j in range(num_boundary_profiles):
            outranking_relation_matrix[0, i, j] = (
                credibility_index[0, i, j] >= credibility_threshold
            )
            outranking_relation_matrix[1, i, j] = (
                credibility_index[1, i, j] >= credibility_threshold
            )

    return outranking_relation_matrix

### Task 9 (maximum points: 1)

Implement the `calculate_pessimistic_assigment` function.

Use the pessimistic rule: assign each alternative to the highest category for which the alternative is at least as good as the boundary profile.


In [18]:
# CHECK EXACT
def calculate_pessimistic_assigment(relation: pd.DataFrame) -> pd.DataFrame:
    """
    Function that calculates pessimistic assigment for given relation between alternatives and boundary profiles

    :param relation: pandas dataframe with relation between alternatives as rows and boundary profiles as columns. With "<" or ">" for preference, "I" for indifference and "?" for incompatibility
    :return: dataframe with pessimistic assigment
    """
    num_alternatives = relation.shape[0]
    num_boundary_profiles = relation.shape[1]

    # Categories are defined by boundary profiles: b1..bm -> C1..C(m+1)
    categories = [f"C{i}" for i in range(1, num_boundary_profiles + 2)]
    assignment = pd.DataFrame(index=relation.index, columns=["Pessimistic Assignment"])

    for i in range(num_alternatives):
        assigned_category = categories[0]
        # Go from right: skip '<' and '?', stop at 'I' or '>'
        for j in range(num_boundary_profiles - 1, -1, -1):
            symbol = relation.iloc[i, j]
            if symbol in ["I", ">"]:
                assigned_category = categories[j + 1]
                break
        assignment.iloc[i, 0] = assigned_category

    return assignment

### Task 10 (maximum points: 1)

Implement the `calculate_optimistic_assigment` function.

Use the optimistic rule: assign each alternative to the lowest category whose upper boundary profile does not sufficiently outrank the alternative.


In [19]:
# TODO
# CHECK EXACT
def calculate_optimistic_assigment(relation: pd.DataFrame) -> pd.DataFrame:
    """
    Function that calculates optimistic assigment for given relation between alternatives and boundary profiles

    :param relation: pandas dataframe with relation between alternatives as rows and boundary profiles as columns. With "<" or ">" for preference, "I" for indifference and "?" for incompatibility
    :return: dataframe with optimistic assigment
    """
    num_alternatives = relation.shape[0]
    num_boundary_profiles = relation.shape[1]

    # Categories are defined by boundary profiles: b1..bm -> C1..C(m+1)
    categories = [f"C{i}" for i in range(1, num_boundary_profiles + 2)]
    assignment = pd.DataFrame(index=relation.index, columns=["Optimistic Assignment"])

    for i in range(num_alternatives):
        assigned_category = categories[-1]
        # Go from left: skip '>', 'I' and '?', stop at '<'
        for j in range(num_boundary_profiles):
            symbol = relation.iloc[i, j]
            if symbol == "<":
                assigned_category = categories[j]
                break
        assignment.iloc[i, 0] = assigned_category

    return assignment

In [20]:
# Build full ELECTRE TRI-B pipeline outputs needed for assignment
marginal_concordance_matrix = calculate_marginal_concordance_matrix(
    dataset, boundary_profiles, preference_information
)
comprehensive_concordance_index = calculate_comprehensive_concordance_matrix(
    marginal_concordance_matrix, preference_information
)
marginal_discordance_matrix = calculate_marginal_discordance_matrix(
    dataset,
    boundary_profiles,
    preference_information[["p"]],
    preference_information[["v"]],
    preference_information[["type"]],
)
credibility_index = calculate_credibility_index(
    comprehensive_concordance_index, marginal_discordance_matrix
)
outranking_relation_matrix = calculate_outranking_relation_matrix(
    credibility_index, credibility_threshold
)

# Create relation table with symbols: >, <, I, ?
relation = pd.DataFrame(index=dataset.index, columns=boundary_profiles.index)
for i, alt in enumerate(dataset.index):
    for j, profile in enumerate(boundary_profiles.index):
        a_outranks_b = outranking_relation_matrix[0, i, j]
        b_outranks_a = outranking_relation_matrix[1, i, j]
        if a_outranks_b and b_outranks_a:
            relation.loc[alt, profile] = "I"
        elif a_outranks_b and not b_outranks_a:
            relation.loc[alt, profile] = ">"
        elif not a_outranks_b and b_outranks_a:
            relation.loc[alt, profile] = "<"
        else:
            relation.loc[alt, profile] = "?"

pessimistic_assignment = calculate_pessimistic_assigment(relation)
optimistic_assignment = calculate_optimistic_assigment(relation)

print("Relation matrix:")
display(relation)
print("Assignments:")
display(pd.concat([pessimistic_assignment, optimistic_assignment], axis=1))

Relation matrix:


Alternative,b1,b2
Alternative,,
a1,>,>
a2,?,<
a3,>,>
a4,>,<
a5,<,<
a6,I,<
a7,>,<
a8,?,<


Assignments:


,Pessimistic Assignment,Optimistic Assignment
Alternative,,
a1,C3,C3
a2,C1,C2
a3,C3,C3
a4,C2,C2
a5,C1,C1
a6,C2,C2
a7,C2,C2
a8,C1,C2


## HOMEWORK

**Deadline:** 29.04.2026, 23:59

Using your implementations of the **PROMETHEE** and **ELECTRE** methods prepared during the laboratory sessions, perform an analysis of the dataset provided for the UTA laboratories.

As a solution, submit:

- Jupyter notebooks for:
  - PROMETHEE (max 7 points)
  - ELECTRE (max 10 points)
- A report in **PDF format**, in which you answer the questions listed below.

### PROMETHEE (max points: 10)

- Describe the preference information used as input to the method. _(2.0 points)_
- Present the final results obtained using the method. _(2.0 points)_
- Compare the complete and partial rankings. _(2.0 points)_
- Discuss your findings regarding the method, including:
  - the performance of the best and worst alternatives,
  - whether the pairwise comparisons defined in the dataset report are satisfied. _(4.0 points)_

### ELECTRE (max points: 10)

- Describe the preference information used as input to the method. _(2.0 points)_
- Present the final results obtained using the method. _(2.0 points)_
- Compare the optimistic and pessimistic class assignments. _(2.0 points)_
- Discuss your findings regarding the method, including:
  - the performance of the best and worst alternatives,
  - whether the pairwise comparisons defined in the dataset report are satisfied. _(4.0 points)_

### Method Comparison (max points: 3)

- Comment on the similarities and differences between the methods. _(3.0 points)_

**Final grade:**  
The final grade will be based on the total number of points obtained.
